In [ ]:
!apt-get install zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!which ollama && ollama --version

In [ ]:
import os, subprocess, time

# Kill any stale instance (in case you re-run the cell)
subprocess.run(["pkill", "-f", "ollama serve"], stderr=subprocess.DEVNULL)

# Launch in background
log = open("/kaggle/working/ollama.log", "w")
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log, stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"},
    start_new_session=True,   # detach from notebook process group
)

# Wait until the server is actually accepting connections
import urllib.request
for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print(f"✅ Ollama ready after {i+1}s")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama failed to start — check /kaggle/working/ollama.log")

In [ ]:
!ollama pull qwen3:14b

In [ ]:
!ollama list

In [ ]:
# !curl -s http://127.0.0.1:11434/api/generate -d '{ "model": "mistral-nemo:latest","prompt": "Return JSON with keys greeting and language for: hello in french","format": "json","stream": false }' | python3 -c "import sys,json; r=json.load(sys.stdin); print(r['response'])"

In [ ]:
!pip install -q -U crawl4ai pydantic
!crawl4ai-setup   # downloads Playwright Chromium (~150MB), one-time

In [ ]:
!python -m playwright install chromium
!python -m playwright install-deps

In [ ]:
# Cell 1: Install + Setup (run once)
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

# Start ollama serve as a true background process via Python
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True  # ← this is the key
)

# Wait for server to be ready
import urllib.request
for i in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("✅ Ollama server is running!")
        break
    except:
        time.sleep(1)
else:
    print("❌ Ollama server didn't start")

# Now pull the model
!ollama pull mistral

In [ ]:
import asyncio, json
from pydantic import BaseModel
from typing import List
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode, LLMConfig, LLMExtractionStrategy

class Article(BaseModel):
    title: str
    summary: str
    key_points: List[str]

class ArticleExtraction(BaseModel):
    articles: List[Article]

async def extract():
    strat = LLMExtractionStrategy(
        llm_config=LLMConfig(
            provider="ollama/qwen3:14b",
            base_url="http://localhost:11434",       # ✅ No /v1
        ),
        schema=ArticleExtraction.model_json_schema(),
        extraction_type="schema",
        instruction="Extract the article title, a summary, and key points from this news article.",
        apply_chunking=True,
        chunk_token_threshold=2000,
        overlap_rate=0.1,
        extra_args={"temperature": 0.0, "max_tokens": 3000},
        verbose=True
    )
    cfg = CrawlerRunConfig(extraction_strategy=strat, cache_mode=CacheMode.BYPASS)
    async with AsyncWebCrawler(config=BrowserConfig(headless=True)) as crawler:
        r = await crawler.arun(
            "https://thehackernews.com/2026/07/citrix-patches-six-netscaler-flaws.html",
            config=cfg
        )
        if r.success:
            print(json.dumps(json.loads(r.extracted_content), indent=2))
            strat.show_usage()
        else:
            print("Error:", r.error_message)

await extract()

In [ ]:
requests.get("http://127.0.0.1:8002/nice?prompt=hello")

In [ ]:
class SupportArticle(BaseModel):
    title: str
    summary: str
    affected_software: List[str]       # e.g. ["iOS 18.5", "macOS 15.5"]
    key_information: List[str]         # main points / warnings
    steps: List[str]                   # numbered steps if present

class SupportArticleExtraction(BaseModel):
    articles: List[SupportArticle]

async def extract():
    strat = LLMExtractionStrategy(
        llm_config=LLMConfig(
            provider="ollama/qwen3:14b",
            base_url="http://localhost:11434",
        ),
        schema=SupportArticleExtraction.model_json_schema(),
        extraction_type="schema",
        instruction=(
            "Extract the support article details: title, summary, "
            "affected software/OS versions, key information or warnings, "
            "and any numbered steps from the content."
        ),
        apply_chunking=True,
        chunk_token_threshold=2000,
        overlap_rate=0.1,
        extra_args={"temperature": 0.0, "max_tokens": 3000},
        verbose=True
    )
    cfg = CrawlerRunConfig(extraction_strategy=strat, cache_mode=CacheMode.BYPASS)
    async with AsyncWebCrawler(config=BrowserConfig(headless=True)) as crawler:
        r = await crawler.arun(
            "https://support.apple.com/en-us/127121",
            config=cfg
        )
        if r.success:
            print(json.dumps(json.loads(r.extracted_content), indent=2))
            strat.show_usage()
        else:
            print("Error:", r.error_message)

await extract()

In [ ]:
##ROUGH WORK start here

In [ ]:
!pip install fastapi uvicorn

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

# 1. Initialize your FastAPI app
app = FastAPI()

# 2. Define your endpoints
@app.get("/")
def home():
    return {"status": "success", "message": "Welcome to your Kaggle FastAPI!"}

@app.get("/predict")
def predict(data_input: float):
    # Imagine your ML model prediction happens here
    prediction = data_input * 2 
    return {"input": data_input, "prediction": prediction}

# 3. Create the TestClient
client = TestClient(app)

# 4. Test the endpoints directly in the notebook
print("Testing Root Endpoint:")
response = client.get("/")
print(response.status_code, response.json())

print("\nTesting Predict Endpoint:")
response_predict = client.get("/predict?data_input=42.0")
print(response_predict.status_code, response_predict.json())

In [ ]:
# import threading
# import time
# from fastapi import FastAPI
# import uvicorn

# app = FastAPI()

# @app.get("/")
# def read_root():
#     return {"hello": "from the background thread!"}

# @app.get("/square/{number}")
# def square_number(number: int):
#     return {"original": number, "squared": number ** 2}

# # Function to run uvicorn
# def run_server():
#     # We use port 8000 on localhost
#     uvicorn.run(app, host="0.0.0.0", port=8000)

# # Launch the server in a separate thread
# server_thread = threading.Thread(target=run_server, daemon=True)
# server_thread.start()

# # Give the server a second to wake up
# time.sleep(1)
# print("Server is up and running in the background!")



from fastapi import FastAPI
from fastapi.testclient import TestClient
import requests

app = FastAPI()

@app.get("/nice")
def generate(prompt: str):

    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen3:14b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()

client = TestClient(app)

response = client.get(
    "/nice",
    params={"prompt": "What is CAN FD?"}
)

print(response.status_code)
print(response.json())


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
import requests
import threading
import uvicorn

app = FastAPI()

@app.get("/nice")
def generate(prompt: str):

    response = requests.post(
        "http://127.0.0.1:11434/api/generate",
        json={
            "model": "qwen3:14b",
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8002)
    
threading.Thread(target=run_server, daemon=True).start()

client = TestClient(app)

# response = client.get(
#     "/nice",
#     params={"prompt": "What is CAN FD?"}
# )

# print(response.status_code)
# print(response.json())


In [ ]:
!lsof -i :8001

In [ ]:
!ps -ef | grep uvicorn

In [ ]:
import requests

r = requests.get(
    "http://127.0.0.1:8002/nice",
    params={
        "prompt": "what 2+2=?"
    }
)

print(r.status_code)
print(r.json())

In [ ]:
print(
    requests.get(
        "http://127.0.0.1:8002/openapi.json"
    ).json()["paths"].keys()
)


In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64


In [ ]:
# !./cloudflared-linux-amd64 tunnel --url http://localhost:8002

In [ ]:
import subprocess
import threading

def run_cloudflare():
    process = subprocess.Popen(
        [
            "./cloudflared-linux-amd64",
            "tunnel",
            "--url",
            "http://localhost:8002"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    for line in process.stdout:
        print("[Cloudflare]", line.strip())

threading.Thread(
    target=run_cloudflare,
    daemon=True
).start()
